In [7]:
import sys
import os

PROJECT_ROOT = os.path.abspath("../..")
sys.path.append(PROJECT_ROOT)

import time
import pandas as pd
from Scripts.corss_validation.cross_val import cross_validate
from Scripts.models.random_forest import RandomForestModel

In [2]:
# load data
df = pd.read_csv("../../Data/pre_split_features.csv")
# dataset sampling for faster execution during development
df = df.sample(n=4000,random_state=42).reset_index(drop=True)

In [3]:
param_grid = {
    "max_depth": [5, 10, 20]
}

start_time = time.time()
print("\n=== Base engineered features ===")
results = cross_validate(
    model_class=RandomForestModel,
    df=df,
    param_grid=param_grid,
    n_outer=3,
    n_inner=2,
    scoring="f1_macro",
    use_interactions=False
)
elapsed = time.time() - start_time
print(f"Execution time: {elapsed:.2f} sec")



=== Base engineered features ===
Samples: 4000, Features: 74
fold 1/3 f1_macro=0.3055 params={'max_depth': 5}
fold 2/3 f1_macro=0.2595 params={'max_depth': 5}
fold 3/3 f1_macro=0.2740 params={'max_depth': 5}

results:
  accuracy: 0.4417 +/- 0.0164
  precision_macro: 0.2838 +/- 0.0136
  recall_macro: 0.2995 +/- 0.0129
  f1_macro: 0.2797 +/- 0.0192
  train_f1: 0.5293 +/- 0.0252
  train_test_gap: 0.2496 +/- 0.0132
Execution time: 5.04 sec


In [4]:
start_time = time.time()

print("\n=== Enhanced behavioural features ===")
results_enhanced = cross_validate(
    model_class=RandomForestModel,
    df=df,
    param_grid=param_grid,
    n_outer=3,
    n_inner=2,
    scoring="f1_macro",
    use_interactions=True
)

elapsed = time.time() - start_time
print(f"Execution time: {elapsed:.2f} sec")


=== Enhanced behavioural features ===
Samples: 4000, Features: 100
fold 1/3 f1_macro=0.3159 params={'max_depth': 5}
fold 2/3 f1_macro=0.2837 params={'max_depth': 5}
fold 3/3 f1_macro=0.2944 params={'max_depth': 5}

results:
  accuracy: 0.4607 +/- 0.0192
  precision_macro: 0.3005 +/- 0.0093
  recall_macro: 0.3116 +/- 0.0115
  f1_macro: 0.2980 +/- 0.0134
  train_f1: 0.5519 +/- 0.0050
  train_test_gap: 0.2539 +/- 0.0160
Execution time: 7.42 sec


In [6]:
# save results
pd.DataFrame(results["fold_metrics"]).to_csv("../../Results/fold_metrics_rf.csv", index=False)
pd.DataFrame(results["best_params"]).to_csv("../../Results/best_params_rf.csv", index=False)

pd.DataFrame(results_enhanced["fold_metrics"]).to_csv("../../Results/fold_metrics_rf_enhanced.csv", index=False)
pd.DataFrame(results_enhanced["best_params"]).to_csv("../../Results/best_params_rf_enhanced.csv", index=False)

print("\nClass distribution:")
print(df["review_score"].value_counts(normalize=True))


Class distribution:
review_score
5    0.59575
4    0.20075
1    0.09600
3    0.07675
2    0.03075
Name: proportion, dtype: float64
